In [24]:
%reload_ext autoreload
%autoreload 2
# notebooks/result_processing.ipynb

# ---
# # Experiment Results Analysis
# This notebook orchestrates the loading, analysis, validation, and reporting 
# of experiment results logged to Weights & Biases.
# ---

# ## 1. Setup and Imports
# Import necessary libraries and custom modules. The `sys.path.append` line
# allows the notebook to find the custom `lmetk_report` package.
import sys
import pandas as pd
sys.path.append('..')

from lmetk_report.run_processor import load_runs_df
from lmetk_report.analysis import analyze_results
from lmetk_report.reporting import (
    create_summary_table,
    display_experiment_stats,
    display_performance_tables,
    display_hp_impact
)

# ---
# ## 2. Configuration
# Define all key parameters for the analysis in one place.
# ---
PROJECT_NAME = "unibo_full"
USER = "imos-buildingenergy-1"

# Metrics to extract
METRIC_PREFIXES = ["val/", "test/", "test_best_rerun/", "val_best_rerun/"]
REPORTING_METRICS = ["loss", "mse", "mae", "rmse", "f1", "accuracy"]

# --- NEW: Scientific Rigor Settings ---
# If defined, the analysis will DROP any configuration group that does not 
# contain exactly these seeds. Set to `None` to include all available data.

# Data seeds are set to None (Flexible mode)
REQUIRED_DATA_SEEDS = None      

# Model seeds are strictly enforced (Strict mode)
REQUIRED_MODEL_SEEDS = {72, 88, 101, 666, 226688}

# Columns to ignore if they vary within a group (avoids "Hidden Variation" warnings)
ADDITIONAL_IGNORED_COLS = ["gpu_id", "slurm_job_id"] 

# Configure pandas display options for better readability in the notebook.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---
# ## 3. Data Loading
# Fetch data from wandb (or a local cache) and prepare the initial DataFrame.
# The `load_runs_df` function handles all the complex cleaning and flattening.
# ---
print("🚀 Loading Run Data...")
df, config_columns, dropped_columns = load_runs_df(
    project_name=PROJECT_NAME, 
    user=USER
)
print(f"Loaded {len(df)} runs.")

# ---
# ## 4. Core Analysis
# Run the main analysis function. This now includes:
# 1. Grouping by Model + Hyperparameters
# 2. Strict Seed Validation (Filtering incomplete groups)
# 3. Hidden Variation Detection (Warning about silent bugs)
# 4. Aggregation & Best Model Selection
# ---
print("\n🔬 Analyzing Results & Validating Consistency...")

all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    dropped_columns=dropped_columns,
    reporting_metrics=REPORTING_METRICS,
    metric_prefixes=METRIC_PREFIXES,
    # --- New Validation Arguments ---
    required_data_seeds=REQUIRED_DATA_SEEDS,
    required_model_seeds=REQUIRED_MODEL_SEEDS,
    additional_ignored_cols=ADDITIONAL_IGNORED_COLS
)

# ---
# ## 5. Reporting & Visualizations
# ---

# ### A. Experiment Statistics
# Shows how many configurations completed successfully and 
# confirms which seeds were actually found in the data.
display_experiment_stats(all_results)

# ### B. Performance Leaderboards
# Displays a separate, clean pivot table for every metric found (MSE, MAE, etc.).
display_performance_tables(all_results)

# ### C. Hyperparameter Impact Analysis
# Shows the top 10 configurations for each model, exposing exactly 
# which hyperparameters led to the best results.
display_hp_impact(all_results)

# ### D. Legacy Summary Table
# A single flattened table containing all metrics (useful for copying to Excel/LaTeX).
print("\n📊 Full Summary Table:")
summary_df = create_summary_table(all_results)
display(summary_df)

🚀 Loading Run Data...
Loaded 90 runs from cached file: csv_files/unibo_full.csv
Dropped 27 columns with >90% NaN values.
The columns were drropped: ['d_x_mask', 'ckpt_path', 'profiler_cfg', 'subset_range', 'val_subset_ratio', 'optimizer_factory', 'scheduler_factory', 'test_subset_ratio', 'train_subset_ratio', 'freq', 'pad_value', 'null_value', 'target_mask', 'variable_emb_type', 'model.freq', 'datamodule.val_subset_ratio', 'datamodule.test_subset_ratio', 'datamodule.train_subset_ratio', 'callbacks.early_stopping.stopping_threshold', 'callbacks.early_stopping.divergence_threshold', 'callbacks.early_stopping.check_on_train_epoch_end', 'callbacks.model_checkpoint.every_n_epochs', 'callbacks.model_checkpoint.every_n_train_steps', 'callbacks.model_checkpoint.train_time_interval', 'callbacks.model_checkpoint.save_on_train_epoch_end', 'evaluator._evaluator_defaults.inverse_transform_name', 'logger.wandb.anonymous']
Dropped 60 columns based on name filters (e.g., 'paths.').
Loaded 90 runs.

🔬 

,Dataset,Model,Configs Completed,Seeds Found
0,UNIBO21,lmetk.baselines.timeseries_transformer_model.Timeseries_Transformer_Forecaster,2,Data: ? | Model: ?
1,UNIBO21,lmetk.baselines.patchtst_model.PatchTST_Forecaster,2,Data: ? | Model: ?
2,UNIBO21,lmetk.baselines.crossformer_model.Crossformer_Forecaster,1,Data: ? | Model: ?
3,UNIBO21,lmetk.baselines.lstm_model.LSTM_Forecaster,2,Data: ? | Model: ?
4,UNIBO21,lmetk.baselines.tide_model.TiDE_Forecaster,2,Data: ? | Model: ?
5,UNIBO21,lmetk.model.wrappers.fit_predict_xgboost_wrapper.FitPredictXGBoostWrapper,2,Data: ? | Model: ?
6,UNIBO21,lmetk.model.wrappers.fit_predict_tabpfn_wrapper.FitPredictTabPFNWrapper,10,Data: ? | Model: ?
7,UNIBO21,lmetk.model.wrappers.fit_predict_tabdpt_wrapper.FitPredictTabDPTWrapper,10,Data: ? | Model: ?
8,UNIBO21,lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,2,Data: ? | Model: ?



  METRIC: test/loss


Dataset,UNIBO21
Model,
lmetk.baselines.crossformer_model.Crossformer_Forecaster,0.01 ± 0.00
lmetk.baselines.lstm_model.LSTM_Forecaster,0.01 ± 0.00
lmetk.baselines.patchtst_model.PatchTST_Forecaster,0.06 ± 0.00
lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster,0.02 ± 0.00
lmetk.baselines.tide_model.TiDE_Forecaster,0.06 ± 0.00
lmetk.baselines.timeseries_transformer_model.Timeseries_Transformer_Forecaster,0.01 ± 0.00
lmetk.model.wrappers.fit_predict_tabdpt_wrapper.FitPredictTabDPTWrapper,1.00 ± nan
lmetk.model.wrappers.fit_predict_tabpfn_wrapper.FitPredictTabPFNWrapper,1.00 ± nan
lmetk.model.wrappers.fit_predict_xgboost_wrapper.FitPredictXGBoostWrapper,1.00 ± 0.00



  METRIC: test/mae_mean


...

,task_definition.seq_len,model.random_state,cache.use_preprocessing_file_lock,val/rmse_mean_mean,val_best_rerun/rmse_mean_mean,val/mae_mean_mean,test_best_rerun/phm_score_mean_mean,val/mse_mean_mean,test_best_rerun/loss_mean,val/phm_score_mean_mean,test/mape_mean_mean,test_best_rerun/mse_mean_mean,val_best_rerun/loss_mean,test/mse_mean_mean,val/mape_mean_mean,test_best_rerun/rmse_mean_mean,val_best_rerun/phm_score_mean_mean,val/loss_mean,val/loss_std,val/loss_count,val_best_rerun/mae_mean_mean,test_best_rerun/mape_mean_mean,val_best_rerun/mape_mean_mean,test/mae_mean_mean,val_best_rerun/mse_mean_mean,test/phm_score_mean_mean,test/rmse_mean_mean,test/loss_mean,test_best_rerun/mae_mean_mean
0,1,72.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.408674,NaN,NaN,1523.797607,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,17.801983,NaN,0.213903,24.112976,1.0,NaN
1,1,88.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.351775,NaN,NaN,1473.502930,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,17.159298,NaN,0.232765,22.988939,1.0,NaN
2,1,101.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.405824,NaN,NaN,1528.483154,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,18.043541,NaN,0.190569,24.064379,1.0,NaN
3,1,666.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.332183,NaN,NaN,1445.523560,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,16.997547,NaN,0.220066,22.939529,1.0,NaN
4,1,226688.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.342143,NaN,NaN,1477.128784,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,17.169874,NaN,0.224742,23.168985,1.0,NaN
5,5,72.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.276140,NaN,NaN,2931.019531,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,34.060406,NaN,0.086204,45.252125,1.0,NaN
6,5,88.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.076589,NaN,NaN,2566.898193,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,30.801632,NaN,0.096684,40.832153,1.0,NaN
7,5,101.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.355346,NaN,NaN,2933.917236,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,34.750423,NaN,0.089881,45.905918,1.0,NaN
8,5,666.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.093873,NaN,NaN,2715.267090,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,31.496929,NaN,0.089824,42.562515,1.0,NaN
9,5,226688.0,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.424437,NaN,NaN,2789.009766,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,33.910114,NaN,0.098670,44.230572,1.0,NaN



>>> lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster on UNIBO21
    (Sorted by val/loss)


,datamodule.train_batch_size,val/rmse_mean_mean,val_best_rerun/rmse_mean_mean,val/mae_mean_mean,test_best_rerun/phm_score_mean_mean,val/mse_mean_mean,test_best_rerun/loss_mean,val/phm_score_mean_mean,test/mape_mean_mean,test_best_rerun/mse_mean_mean,val_best_rerun/loss_mean,test/mse_mean_mean,val/mape_mean_mean,test_best_rerun/rmse_mean_mean,val_best_rerun/phm_score_mean_mean,val/loss_mean,val/loss_std,val/loss_count,val_best_rerun/mae_mean_mean,test_best_rerun/mape_mean_mean,val_best_rerun/mape_mean_mean,test/mae_mean_mean,val_best_rerun/mse_mean_mean,test/phm_score_mean_mean,test/rmse_mean_mean,test/loss_mean,test_best_rerun/mae_mean_mean
1,512,49.312342,37.937078,36.507716,0.118912,3759.240723,0.017805,0.060453,1.307436,3865.100146,0.010327,3865.100146,0.877673,50.051400,0.081614,0.019194,0.003985,5,27.557526,1.307436,0.848317,40.081081,1821.608374,0.118912,50.051400,0.017805,40.081081
0,16,67.065514,42.229240,45.342693,0.099966,7207.883691,0.024888,0.084966,1.214623,5213.605859,0.011080,5213.605859,1.051397,57.420308,0.074970,0.032588,0.004574,5,30.875090,1.214623,1.180502,43.254569,2160.925244,0.099966,57.420308,0.024888,43.254569



📊 Full Summary Table:


Dataset                                                           UNIBO21                                                                                                                             \
Metric                                                          test/loss           test/mae_mean         test/mape_mean                test/mse_mean    test/phm_score_mean          test/rmse_mean   
Model                                                                                                                                                                                                  
lmetk.baselines.crossformer_model.Crossformer_F...  0.0097 ± 0.0048 (n=5)  30.6444 ± 6.4743 (n=5)  1.0162 ± 0.4488 (n=5)   1971.4839 ± 736.6324 (n=5)  0.1444 ± 0.0199 (n=5)  38.1461 ± 7.7843 (n=5)   
lmetk.baselines.lstm_model.LSTM_Forecaster          0.0110 ± 0.0024 (n=5)  33.3621 ± 4.1465 (n=5)  1.2551 ± 0.4075 (n=5)   2799.1691 ± 560.7189 (n=5)  0.1360 ± 0.0122 (n=5)  42.8255 ± 5.0662 (n=5)   
lmetk.baselines.patchtst_model.PatchTST_Forecaster  0.0563 ± 0.0001 (n=5)  76.0613 ± 0.1921 (n=5)  1.9116 ± 0.0155 (n=5)    11467.1777 ± 5.7352 (n=5)  0.0910 ± 0.0001 (n=5)  89.6905 ± 0.1512 (n=5)   
lmetk.baselines.spacetimeformer_model.Spacetime...  0.0178 ± 0.0048 (n=5)  40.0811 ± 7.5146 (n=5)  1.3074 ± 0.3448 (n=5)  3865.1001 ± 1068.6115 (n=5)  0.1189 ± 0.0229 (n=5)  50.0514 ± 8.6885 (n=5)   
lmetk.baselines.tide_model.TiDE_Forecaster          0.0570 ± 0.0000 (n=5)  74.2821 ± 0.0248 (n=5)  1.7637 ± 0.0021 (n=5)    11551.7504 ± 1.6556 (n=5)  0.0900 ± 0.0000 (n=5)  88.3522 ± 0.0177 (n=5)   
lmetk.baselines.timeseries_transformer_model.Ti...  0.0126 ± 0.0026 (n=5)  30.4111 ± 3.8715 (n=5)  1.0601 ± 0.3807 (n=5)   2883.6320 ± 546.3463 (n=5)  0.1372 ± 0.0248 (n=5)  38.5565 ± 4.5398 (n=5)   
lmetk.model.wrappers.fit_predict_tabdpt_wrapper...     1.0000 ± nan (n=1)     17.8020 ± nan (n=1)     0.4087 ± nan (n=1)        1523.7976 ± nan (n=1)     0.2139 ± nan (n=1)     24.1130 ± nan (n=1)   
lmetk.model.wrappers.fit_predict_tabpfn_wrapper...     1.0000 ± nan (n=1)     16.1807 ± nan (n=1)     0.2391 ± nan (n=1)        1530.0608 ± nan (n=1)     0.2192 ± nan (n=1)     21.4070 ± nan (n=1)   
lmetk.model.wrappers.fit_predict_xgboost_wrappe...  1.0000 ± 0.0000 (n=5)  19.5681 ± 0.0000 (n=5)  0.4399 ± 0.0000 (n=5)     1307.9506 ± 0.0000 (n=5)  0.2060 ± 0.0000 (n=5)  25.6761 ± 0.0000 (n=5)   

Dataset                                                                                                                                                                                   \
Metric                                               test_best_rerun/loss test_best_rerun/mae_mean test_best_rerun/mape_mean     test_best_rerun/mse_mean test_best_rerun/phm_score_mean   
Model                                                                                                                                                                                      
lmetk.baselines.crossformer_model.Crossformer_F...  0.0097 ± 0.0048 (n=5)   30.6444 ± 6.4743 (n=5)     1.0162 ± 0.4488 (n=5)   1971.4839 ± 736.6324 (n=5)          0.1444 ± 0.0199 (n=5)   
lmetk.baselines.lstm_model.LSTM_Forecaster          0.0110 ± 0.0024 (n=5)   33.3621 ± 4.1465 (n=5)     1.2551 ± 0.4075 (n=5)   2799.1691 ± 560.7189 (n=5)          0.1360 ± 0.0122 (n=5)   
lmetk.baselines.patchtst_model.PatchTST_Forecaster  0.0563 ± 0.0001 (n=5)   76.0613 ± 0.1921 (n=5)     1.9116 ± 0.0155 (n=5)    11467.1777 ± 5.7352 (n=5)          0.0910 ± 0.0001 (n=5)   
lmetk.baselines.spacetimeformer_model.Spacetime...  0.0178 ± 0.0048 (n=5)   40.0811 ± 7.5146 (n=5)     1.3074 ± 0.3448 (n=5)  3865.1001 ± 1068.6115 (n=5)          0.1189 ± 0.0229 (n=5)   
lmetk.baselines.tide_model.TiDE_Forecaster          0.0570 ± 0.0000 (n=5)   74.2821 ± 0.0248 (n=5)     1.7637 ± 0.0021 (n=5)    11551.7504 ± 1.6556 (n=5)          0.0900 ± 0.0000 (n=5)   
lmetk.baselines.timeseries_transformer_model.Ti...  0.0126 ± 0.0026 (n=5)   30.4111 ± 3.8715 (n

In [22]:
# 1. Access the raw data for the specific model and dataset
model_key = 'lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster'
dataset_key = 'UNIBO21'

# This DF contains all runs for this model before aggregation
df_subset = all_results[dataset_key][model_key]['non_aggregated_df']

# 2. Filter for ONE specific configuration that shows count=15
# (Based on your screenshot: lr=0.0001, seq_len=10, batch_size=16)
mask = (
    (df_subset['optimization.lr'] == 0.0001) & 
    (df_subset['task_definition.seq_len'] == 10) & 
    (df_subset['datamodule.train_batch_size'] == 16)
)
group_runs = df_subset[mask]

# 3. Display identifiers to spot the duplicates
# 'created_at' will show you if these runs are from different times/days
cols_to_inspect = ['run_name', 'id', 'created_at', 'seed', 'datasource.parameters.data_seed']
valid_cols = [c for c in cols_to_inspect if c in group_runs.columns]

print(f"Found {len(group_runs)} runs for this configuration (Expected 5).")
print("Listing all runs in this group:")
#display(group_runs[valid_cols].sort_values('created_at'))

Found 5 runs for this configuration (Expected 5).
Listing all runs in this group:


In [23]:
print(df_subset.iloc[0].to_latex())

\begin{tabular}{ll}
\toprule
 & 0 \\
\midrule
AvgTime/train_batch_mean & 0.020657 \\
AvgTime/train_batch_std & 0.005162 \\
AvgTime/train_epoch_mean & 11.607952 \\
AvgTime/train_epoch_std & 0.450640 \\
AvgTime/val_batch_mean & 0.008616 \\
AvgTime/val_batch_std & 0.000965 \\
AvgTime/val_epoch_mean & 0.036777 \\
AvgTime/val_epoch_std & 0.002732 \\
_runtime & 306 \\
_step & 490 \\
_timestamp & 1769624335.062802 \\
epoch & 25 \\
lr-AdamW & 0.000902 \\
plots/test/pred_vs_target_unit_10 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_10_478_79be3a6b41f2384b3f28.plotly.json', 'sha256': '79be3a6b41f2384b3f284f3a04ea9ec558be424cd78f12632c04666c0f8665d2', 'size': 29913} \\
plots/test/pred_vs_target_unit_11 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_11_485_aa7b2f092bbec702f9dc.plotly.json', 'sha256': 'aa7b2f092bbec702f9dcde687afc921f046ea5c1f5c4e71eeb175f1442ced9cc', 'size': 29299} \\
plots/test/pred_vs_target_unit_13 & {'_type':

In [19]:
print(df_subset.iloc[0].to_latex())

\begin{tabular}{ll}
\toprule
 & 30 \\
\midrule
AvgTime/train_batch_mean & 0.024092 \\
AvgTime/train_batch_std & 0.006863 \\
AvgTime/train_epoch_mean & 12.582578 \\
AvgTime/train_epoch_std & 1.215955 \\
AvgTime/val_batch_mean & 0.018669 \\
AvgTime/val_batch_std & 0.008642 \\
AvgTime/val_epoch_mean & 0.073752 \\
AvgTime/val_epoch_std & 0.012027 \\
_runtime & 1358 \\
_step & 1918 \\
_timestamp & 1762901279.126953 \\
epoch & 100 \\
lr-Adam & NaN \\
plots/test/pred_vs_target_unit_10 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_10_1906_ce34dedb41d76ea75180.plotly.json', 'sha256': 'ce34dedb41d76ea75180778ff341f8960afc2147b56aaece910ba30348b5591a', 'size': 28982} \\
plots/test/pred_vs_target_unit_11 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_11_1913_09482285d5caa3b46983.plotly.json', 'sha256': '09482285d5caa3b469839a8d5678ab7aeeee53c83d349bc317ef18a2e1c4c76e', 'size': 28379} \\
plots/test/pred_vs_target_unit_13 & {'_type':

In [12]:
import pandas as pd

# 1. Isolate the specific group
target_model = 'lmetk.baselines.spacetimeformer_model.Spacetimeformer_Forecaster'

# Update this mask to match the specific "duplicate" group you saw in your table
mask = (
    (df['model._target_'] == target_model) &
    (df['optimization.lr'] == 0.0001) & 
    (df['task_definition.seq_len'] == 10) & 
    (df['datamodule.train_batch_size'] == 16)
)
group_runs = df[mask].copy()

# 2. Find "Hidden Variables" (Robust to lists/dicts)
varying_cols = []

for col in group_runs.columns:
    try:
        # Standard check for hashable types (int, float, str)
        if group_runs[col].nunique() > 1:
            varying_cols.append(col)
    except TypeError:
        # Fallback for unhashable types (lists, dicts): Convert to string first
        if group_runs[col].astype(str).nunique() > 1:
            varying_cols.append(col)

# 3. Add identifiers for context
# 'created_at' or '_timestamp' is CRITICAL to see if these are old vs new runs
context_cols = ['run_name', 'seed', 'created_at', '_timestamp'] 

for c in context_cols:
    if c in group_runs.columns and c not in varying_cols:
        varying_cols.insert(0, c)

# 4. Display sorted by Seed to compare side-by-side
print(f"Found {len(group_runs)} runs in this group.")
print("Showing ONLY columns that differ between these runs:")

if not varying_cols:
    print("No varying columns found! These rows are EXACT duplicates.")
else:
    display(
        group_runs[varying_cols]
        .sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])
        .style.set_properties(**{'text-align': 'left'})
    )

Found 0 runs in this group.
Showing ONLY columns that differ between these runs:


,_timestamp,seed,run_name


In [6]:
group_runs.sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])['val_best_rerun/rmse_denormalized_mean']

25    85.029541
26    88.634254
27    85.144997
28    88.849060
29    93.493942
Name: val_best_rerun/rmse_denormalized_mean, dtype: float64

In [7]:
group_runs.sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])[[i for i in group_runs.columns if 'val_best_rerun' in i]]

,val_best_rerun/loss,val_best_rerun/mae_15_denormalized,val_best_rerun/mae_15_normalized,val_best_rerun/mae_18_denormalized,val_best_rerun/mae_18_normalized,val_best_rerun/mae_23_denormalized,val_best_rerun/mae_23_normalized,val_best_rerun/mae_27_denormalized,val_best_rerun/mae_27_normalized,val_best_rerun/mae_5_denormalized,val_best_rerun/mae_5_normalized,val_best_rerun/mae_7_denormalized,val_best_rerun/mae_7_normalized,val_best_rerun/mae_denormalized_mean,val_best_rerun/mae_normalized_mean,val_best_rerun/mape_15_denormalized,val_best_rerun/mape_15_normalized,val_best_rerun/mape_18_denormalized,val_best_rerun/mape_18_normalized,val_best_rerun/mape_23_denormalized,val_best_rerun/mape_23_normalized,val_best_rerun/mape_27_denormalized,val_best_rerun/mape_27_normalized,val_best_rerun/mape_5_denormalized,val_best_rerun/mape_5_normalized,val_best_rerun/mape_7_denormalized,val_best_rerun/mape_7_normalized,val_best_rerun/mape_denormalized_mean,val_best_rerun/mape_normalized_mean,val_best_rerun/mse_15_denormalized,val_best_rerun/mse_15_normalized,val_best_rerun/mse_18_denormalized,val_best_rerun/mse_18_normalized,val_best_rerun/mse_23_denormalized,val_best_rerun/mse_23_normalized,val_best_rerun/mse_27_denormalized,val_best_rerun/mse_27_normalized,val_best_rerun/mse_5_denormalized,val_best_rerun/mse_5_normalized,val_best_rerun/mse_7_denormalized,val_best_rerun/mse_7_normalized,val_best_rerun/mse_denormalized_mean,val_best_rerun/mse_normalized_mean,val_best_rerun/phm_score_15_denormalized,val_best_rerun/phm_score_15_normalized,val_best_rerun/phm_score_18_denormalized,val_best_rerun/phm_score_18_normalized,val_best_rerun/phm_score_23_denormalized,val_best_rerun/phm_score_23_normalized,val_best_rerun/phm_score_27_denormalized,val_best_rerun/phm_score_27_normalized,val_best_rerun/phm_score_5_denormalized,val_best_rerun/phm_score_5_normalized,val_best_rerun/phm_score_7_denormalized,val_best_rerun/phm_score_7_normalized,val_best_rerun/phm_score_denormalized_mean,val_best_rerun/phm_score_normalized_mean,val_best_rerun/rmse_15_denormalized,val_best_rerun/rmse_15_normalized,val_best_rerun/rmse_18_denormalized,val_best_rerun/rmse_18_normalized,val_best_rerun/rmse_23_denormalized,val_best_rerun/rmse_23_normalized,val_best_rerun/rmse_27_denormalized,val_best_rerun/rmse_27_normalized,val_best_rerun/rmse_5_denormalized,val_best_rerun/rmse_5_normalized,val_best_rerun/rmse_7_denormalized,val_best_rerun/rmse_7_normalized,val_best_rerun/rmse_denormalized_mean,val_best_rerun/rmse_normalized_mean
25,0.005646,44.671867,0.041239,36.080811,0.033308,104.976585,0.096910,92.057945,0.084984,60.715847,0.056051,66.602440,0.061485,67.517586,0.062330,1.934772,1.934772,1.588566,1.588566,3.528280,3.528280,6.192298,6.192297,3.645222,3.645222,3.017918,3.017918,3.317842,3.317842,3270.481201,0.002787,4703.015137,0.004008,14702.636719,0.012530,12116.985352,0.010326,5449.311523,0.004644,6282.075684,0.005354,7754.083984,0.006608,0.356089,0.356089,0.373114,0.373114,0.122099,0.122099,0.190011,0.190011,0.226884,0.226884,0.240738,0.240738,0.251489,0.251489,57.188122,0.052794,68.578537,0.063309,121.254425,0.111937,110.077179,0.101619,73.819450,0.068147,79.259544,0.073169,85.029541,0.078496
26,0.006057,38.695671,0.035722,41.855164,0.038639,122.995995,0.113545,103.576431,0.095618,72.726845,0.067139,62.672249,0.057857,73.753723,0.068087,3.003232,3.003231,2.394802,2.394802,3.774184,3.774184,6.207900,6.207900,5.217603,5.217603,3.536097,3.536097,4.022303,4.022303,2361.650146,0.002013,4285.586426,0.003652,18346.722656,0.015636,14237.776367,0.012134,7600.750488,0.006478,5744.092285,0.004895,8762.763672,0.007468,0.365468,0.365468,0.289375,0.289375,0.049095,0.049095,0.122567,0.122567,0.183412,0.183412,0.244611,0.244611,0.209088,0.209088,48.596813,0.044863,65.464394,0.060434,135.450073,0.125042,119.322151,0.110154,87.182281,0.080483,75.789787,0.069966,88.634254,0.081824
27,0.006139,49.166866,0.045389,46.119526,0.042576,81.977654,0.075679,75.429947,0.069634,62.865887,0.058035,80.776985,0.074

In [8]:
group_runs.sort_values(by=['seed', 'created_at'] if 'created_at' in group_runs else ['seed'])[[i for i in group_runs.columns if 'test_best_rerun' in i]]

,test_best_rerun/loss,test_best_rerun/mae_16_denormalized,test_best_rerun/mae_16_normalized,test_best_rerun/mae_19_denormalized,test_best_rerun/mae_19_normalized,test_best_rerun/mae_24_denormalized,test_best_rerun/mae_24_normalized,test_best_rerun/mae_28_denormalized,test_best_rerun/mae_28_normalized,test_best_rerun/mae_6_denormalized,test_best_rerun/mae_6_normalized,test_best_rerun/mae_8_denormalized,test_best_rerun/mae_8_normalized,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_16_denormalized,test_best_rerun/mape_16_normalized,test_best_rerun/mape_19_denormalized,test_best_rerun/mape_19_normalized,test_best_rerun/mape_24_denormalized,test_best_rerun/mape_24_normalized,test_best_rerun/mape_28_denormalized,test_best_rerun/mape_28_normalized,test_best_rerun/mape_6_denormalized,test_best_rerun/mape_6_normalized,test_best_rerun/mape_8_denormalized,test_best_rerun/mape_8_normalized,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_16_denormalized,test_best_rerun/mse_16_normalized,test_best_rerun/mse_19_denormalized,test_best_rerun/mse_19_normalized,test_best_rerun/mse_24_denormalized,test_best_rerun/mse_24_normalized,test_best_rerun/mse_28_denormalized,test_best_rerun/mse_28_normalized,test_best_rerun/mse_6_denormalized,test_best_rerun/mse_6_normalized,test_best_rerun/mse_8_denormalized,test_best_rerun/mse_8_normalized,test_best_rerun/mse_denormalized_mean,test_best_rerun/mse_normalized_mean,test_best_rerun/phm_score_16_denormalized,test_best_rerun/phm_score_16_normalized,test_best_rerun/phm_score_19_denormalized,test_best_rerun/phm_score_19_normalized,test_best_rerun/phm_score_24_denormalized,test_best_rerun/phm_score_24_normalized,test_best_rerun/phm_score_28_denormalized,test_best_rerun/phm_score_28_normalized,test_best_rerun/phm_score_6_denormalized,test_best_rerun/phm_score_6_normalized,test_best_rerun/phm_score_8_denormalized,test_best_rerun/phm_score_8_normalized,test_best_rerun/phm_score_denormalized_mean,test_best_rerun/phm_score_normalized_mean,test_best_rerun/rmse_16_denormalized,test_best_rerun/rmse_16_normalized,test_best_rerun/rmse_19_denormalized,test_best_rerun/rmse_19_normalized,test_best_rerun/rmse_24_denormalized,test_best_rerun/rmse_24_normalized,test_best_rerun/rmse_28_denormalized,test_best_rerun/rmse_28_normalized,test_best_rerun/rmse_6_denormalized,test_best_rerun/rmse_6_normalized,test_best_rerun/rmse_8_denormalized,test_best_rerun/rmse_8_normalized,test_best_rerun/rmse_denormalized_mean,test_best_rerun/rmse_normalized_mean
25,0.005065,30.792141,0.028426,30.798347,0.028432,87.903671,0.081149,81.827187,0.075540,128.666290,0.118780,59.390087,0.054827,69.896286,0.064526,1.651149,1.651149,4.698690,4.698690,3.989303,3.989303,4.889970,4.889970,5.925451,5.925451,2.817622,2.817622,3.995364,3.995364,1680.807007,0.001432,1605.695190,0.001368,10669.673828,0.009093,9461.178711,0.008063,18545.105469,0.015805,5336.435059,0.004548,7883.149902,0.006718,0.314925,0.314925,0.382616,0.382616,0.168841,0.168841,0.184514,0.184514,0.006502,0.006502,0.269012,0.269012,0.221069,0.221069,40.997646,0.037847,40.071125,0.036992,103.294113,0.095357,97.268585,0.089795,136.180420,0.125717,73.050903,0.067438,81.810463,0.075524
26,0.005970,38.689022,0.035716,33.813931,0.031216,96.416992,0.089008,84.515388,0.078021,156.255219,0.144249,57.150345,0.052759,77.806816,0.071828,2.553461,2.553461,6.677299,6.677299,4.178485,4.178485,4.508988,4.508988,6.977624,6.977624,3.346066,3.346066,4.706987,4.706987,2657.914551,0.002265,1772.029297,0.001510,12418.380859,0.010583,9471.228516,0.008072,27156.468750,0.023144,4989.267090,0.004252,9744.214844,0.008304,0.254609,0.254609,0.307770,0.307770,0.136200,0.136200,0.130625,0.130625,0.001642,0.001642,0.271274,0.271274,0.183687,0.183687,51.554966,0.047594,42.095478,0.038861,111.437790,0.102875,97.320236,0.089842,164.792206,0.152130,70.634743,0.065207,89.639236,0.082752
27,0.004513,26.860533,0.024797,39.791023,0.036734,77